# ex05 · 自动微分（对应教材 2.4 微积分 + 2.5 自动微分）

> 自动微分是深度学习训练的**心脏**——后面所有训练都是「前向算损失 + 反向算梯度 + 更新参数」。
> 这组题每个都要**先手推导数，再运行验证**，让手推和代码对上。
> 答案在 `solutions/ex05-答案.md`。

In [2]:
import torch

## 题 1 🌱 验证 y = 2·⟨x, x⟩ 的梯度

`x = torch.arange(4.0, requires_grad=True)`，`y = 2 * torch.dot(x, x)`。

手推：y = 2(x₀² + x₁² + x₂² + x₃²)，所以 ∂y/∂xᵢ = [4x0, 4x1, 4x2, 4x3]（提示：对 xᵢ 求导得 4xᵢ）

预测 `x.grad` 的值 = 4x（应该是 [0, 4, 8, 12]）

In [3]:
x = torch.arange(4.0, requires_grad=True)
print('x =', x)
y = 2 * torch.dot(x, x)
y.backward()
print('x.grad =', x.grad)          # 应该等于 4x = [0, 4, 8, 12]
print('验证: x.grad == 4*x →', x.grad == 4 * x)

x = tensor([0., 1., 2., 3.], requires_grad=True)
x.grad = tensor([ 0.,  4.,  8., 12.])
验证: x.grad == 4*x → tensor([True, True, True, True])


## 题 2 🌱 非标量 backward 需要 gradient 参数

先直接运行 `x.grad.zero_()` 清空梯度，再对 `y = x * x`（y 是向量）调用 `y.backward()`，看会发生什么。

然后改成 `y.backward(torch.ones(len(x)))` 再试。手推：d(x²)/dx = 2x，所以 x.grad 应为 ____

In [4]:
x = torch.arange(4.0, requires_grad=True)
y = x * x
try:
    y.backward()
    print('直接 backward 成功，x.grad =', x.grad)
except Exception as e:
    print('直接 backward 报错：', e)

x.grad = None   # 第一次 backward 失败，grad 还是 None（此行仅示范清空方式）
y.backward(torch.ones(len(x)))
print('带参数 backward 后 x.grad =', x.grad)   # 应该等于 2x

直接 backward 报错： grad can be implicitly created only for scalar outputs
带参数 backward 后 x.grad = tensor([0., 2., 4., 6.])


## 题 3 🔧 清空梯度的重要性

梯度会**累加**！预测下面代码第二次 backward 后 `x.grad` 是多少？

> 注意：对**同一张图**连续 backward 会报「图已释放」错（题 5 会讲到）；
> 下面的代码是**重新前向产生新图**再 backward——这正是训练循环里每轮发生的事，梯度因此不断累加。

**思考**：为什么训练循环里每次都要 `optimizer.zero_grad()`？（这是第 3 章的伏笔）

训练循环里每轮都会重新前向产生新图，所以梯度会一轮一轮地累加——这正是必须 `zero_grad()` 的原因。

In [5]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print('第一次 backward 后 x.grad =', x.grad)   # 2x = 4

# 重新前向（产生新图）再 backward —— 训练循环里每轮都是这样
y = x ** 2
y.backward()
print('第二次 backward 后 x.grad =', x.grad)   # 会累加！4 + 4 = 8

# 正确做法：backward 前清空
x.grad.zero_()
y = x ** 2
y.backward()
print('清空后重新 backward，x.grad =', x.grad)

第一次 backward 后 x.grad = tensor(4.)
第二次 backward 后 x.grad = tensor(8.)
清空后重新 backward，x.grad = tensor(4.)


## 题 4 🔧 `detach()` 分离：把变量当常数

`u = y.detach()` 之后，`u` 不再参与梯度计算。手推并预测 `x.grad`：

```python
x = torch.tensor(3.0, requires_grad=True)
y = x * x            # y = 9
u = y.detach()       # u = 9，但是常数，梯度不回溯
z = u * x            # z = 9x，dz/dx = 9
z.sum().backward()
```

预测 x.grad = ____（提示：不是 18，因为 y 到 x 的那条路径被 detach 切断了）

In [11]:
help(torch.detach) #冻结某些量，使之不参与梯度的计算,生成一个副本，原来的量仍然保留梯度

Help on built-in function detach in module torch:

detach(...)



In [13]:
x = torch.tensor(3.0, requires_grad=True)
y = x * x
u = y.detach()
z = u * x
z.sum().backward()
print('x.grad =', x.grad)    # = u 的值 = 9

x.grad = tensor(9.)


## 题 5 🌱 为什么第二次 backward 会报错

运行后回答：报错信息说了什么？计算图默认在 backward 后被**释放**，所以第二次会失败。

改法：`y.backward(retain_graph=True)` 会保留图。试试。

`这里的图指的是正向传播的路径，这是反向传播时的参照`

In [14]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
y.backward()
print('第一次 x.grad =', x.grad)
try:
    y.backward()
except Exception as e:
    print('第二次报错：', e)

# 用 retain_graph=True 保留计算图
x2 = torch.tensor(2.0, requires_grad=True)
y2 = x2 ** 3
y2.backward(retain_graph=True)
y2.backward(retain_graph=True)   # 这次不报错
print('retain_graph 后第二次 x2.grad =', x2.grad)   # 注意：梯度会累加

第一次 x.grad = tensor(12.)
第二次报错： Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.
retain_graph 后第二次 x2.grad = tensor(24.)


## 题 6 🚀 Python 控制流的梯度

```python
def f(a):
    if a * a > 20:      # a=6 时 36>20 为真
        return 2 * a
    else:
        return 3 * a
```

手推：a=6 时走哪个分支？f(6)=____，f'(6)=____（提示：2a 的导数是 2）

In [15]:
def f(a):
    if a * a > 20:
        return 2 * a
    else:
        return 3 * a

a = torch.tensor(6.0, requires_grad=True)
y = f(a)
y.backward()
print('f(6) =', y.item())
print("f'(6) =", a.grad)     # 自动微分只对实际走过的分支求导

f(6) = 12.0
f'(6) = tensor(2.)


## 题 7 🚀 挑战：多项式手推 + 验证

`f(x) = x³ + 2x²`，在 x = 3 处：

1. 手推 f'(x) = ____
2. 手算 f'(3) = ____
3. 用代码验证

In [21]:
# 你的验证代码
x = torch.tensor(3.0, requires_grad=True)
# f(x) = x**3 + 2 * x**2
# 写出来，backward，打印 x.grad
y = x**3 + 2 * x**2
y.backward()
print("梯度：", x.grad)
x.grad.zero_()              #必须清除，否则叠加
u = (x**2).detach()
z = u * x + 2 * u
z.backward()
print("newgrad:", x.grad)

梯度： tensor(39.)
newgrad: tensor(9.)


## 题 8 🚀 挑战：闭卷默写「自动微分三步走」

不看教材，写出任意函数求梯度的完整流程：

1. 创建带 `requires_grad=True` 的张量
2. 构造计算（随便写一个含 x 的表达式，比如 x·sin(x)）
3. `backward()` 并读取 `.grad`

额外：验证 `torch.autograd.grad` 也能得到同样的梯度。

In [12]:
# 你的实现（写在这里）
import torch
x = torch.arange(12.).reshape(3, 4)
x.requires_grad_(True)
y = x*torch.sin(x)
y.backward(torch.ones(3, 4))
print("x.grad = ", x.grad)

x.grad =  tensor([[ 0.0000,  1.3818,  0.0770, -2.8289],
        [-3.3714,  0.4594,  5.4816,  5.9343],
        [-0.1746, -7.7881, -8.9347, -0.9513]])


叶子节点 = 直接创建（grad_fn=None）且 requires_grad=True 的张量；反向传播的梯度只会累积到叶子上，非叶子节点的 .grad 默认为 None（用完即弃、省内存）。所以 x = torch.arange(12, requires_grad=True).reshape(3,4) 里，reshape 是一次运算，结果 x 成了非叶子，梯度其实存到了那个被覆盖、再也拿不到的原始张量上——x.grad 显示 None 不是梯度算错了，而是存错了地方。正确写法是先做完本体运算（reshape 等）再 requires_grad_(True)，保证最终这个张量是叶子。更深一层：view 操作（reshape/transpose/permute）的反向是"无损搬回"（导数=恒等重排，值不变、只换形状），而 sin、矩阵乘这类真变换的反向要乘上自身的导数（链式法则），梯度值会真正多出一个因子（如 sin 多乘 cos）。所以判断"会不会出错"的本质是两条独立的线：.grad 为 None 永远是位置问题，不是数值问题；而 view 无损、真变换多因子，才决定数值是否变化。(计算图有关)

### 此题总结：
1.先进行所有x的本体运算，之后再调用x.requires_grad_(True)

2.sin运算存在torch里面，调用时要加上头头

3.求梯度需要float量，整形不行